In [1]:
# pp.ipynb - preprocess count data and save into formatted adata files.

In [2]:
import anndata as ad
import numpy as np
import os
import pandas as pd
import scanpy as sc

In [3]:
seed_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/seed/rdr_starsolo/HCC3N_600spot.spotxgene.starsolo.h5ad"
rs_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim/rdr_starsolo/HCC3N_simu.spotxgene.starsolo.h5ad"
scrs_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/rdr_starsolo/HCC3N_simu.spotxgene.starsolo.h5ad"

out_dir = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/rs_benchmark/rdr/base/pp"

In [4]:
normal_cell_type = "normal"
tumor_cell_type = "tumor"

# Load Data

In [5]:
os.makedirs(out_dir, exist_ok = True)

### Seed data (STARsolo)

In [6]:
seed = ad.read_h5ad(seed_fn)
seed

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 600 × 33538
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand'

In [7]:
seed.var['feature'] = seed.var['gene']
seed = seed[:, ~seed.var["feature"].duplicated(keep = False)].copy()
seed.obs.index = seed.obs["cell"]
seed.var.index = seed.var["feature"]
seed

AnnData object with n_obs × n_vars = 600 × 33490
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'

In [8]:
seed.X.dtype

dtype('int64')

### scCNASim rs (STARsolo)

In [9]:
rs = ad.read_h5ad(rs_fn)
rs

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 33538
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand'

In [10]:
rs.var['feature'] = rs.var['gene']
rs = rs[:, ~rs.var["feature"].duplicated(keep = False)].copy()
rs.obs.index = rs.obs["cell"]
rs.var.index = rs.var["feature"]
rs

AnnData object with n_obs × n_vars = 1200 × 33490
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'

In [11]:
rs.X.dtype

dtype('int64')

### Only keep chr1-22

In [12]:
target_chroms = [str(i) for i in range(1, 23)]
rs = rs[:, rs.var['chrom'].isin(target_chroms)].copy()
rs

AnnData object with n_obs × n_vars = 1200 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'

### scReadSim (STARsolo)

In [13]:
scrs = ad.read_h5ad(scrs_fn)
scrs

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 33538
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand'

In [14]:
scrs.var['feature'] = scrs.var['gene']
scrs = scrs[:, ~scrs.var["feature"].duplicated(keep = False)].copy()
scrs.obs.index = scrs.obs["cell"]
scrs.var.index = scrs.var["feature"]
scrs

AnnData object with n_obs × n_vars = 1200 × 33490
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'

In [15]:
scrs.X.dtype

dtype('int64')

## Use intersect genes

In [16]:
adata_list = [seed, rs, scrs]

genes = None
for i, adata in enumerate(adata_list):
    if i == 0:
        genes = adata.var["feature"].to_numpy()
    else:
        genes = np.intersect1d(genes, adata.var["feature"])
genes.shape

(32272,)

### Use the same order of genes

In [17]:
seed = seed[:, seed.var["feature"].isin(genes)].copy()
print(seed)

rs = rs[:, seed.var.index].copy()
print(rs)

scrs = scrs[:, seed.var.index].copy()
print(scrs)

AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 1200 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 1200 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'


## Split by cell type

In [18]:
seed_normal = seed[seed.obs["cell_type"] == normal_cell_type, :].copy()
print(seed_normal)

rs_normal = rs[rs.obs["cell_type"] == normal_cell_type, :].copy()
print(rs_normal)

rs_tumor = rs[rs.obs["cell_type"] == tumor_cell_type, :].copy()
print(rs_tumor)

scrs_normal = scrs[scrs.obs["cell_type"] == normal_cell_type, :].copy()
print(scrs_normal)

scrs_tumor = scrs[scrs.obs["cell_type"] == tumor_cell_type, :].copy()
print(scrs_tumor)

AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'


# Save Data

### Check the order of cells and genes

In [19]:
assert np.all(rs_normal.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())
assert np.all(rs_tumor.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())
assert np.all(scrs_normal.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())
assert np.all(scrs_tumor.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())

In [20]:
seed_normal.var.index.name = None
rs_normal.var.index.name = None
rs_tumor.var.index.name = None
scrs_normal.var.index.name = None
scrs_tumor.var.index.name = None

In [21]:
seed_normal.write_h5ad(os.path.join(out_dir, "seed_normal.h5ad"), compression = 'gzip')
rs_normal.write_h5ad(os.path.join(out_dir, "rs_normal.h5ad"), compression = 'gzip')
rs_tumor.write_h5ad(os.path.join(out_dir, "rs_tumor.h5ad"), compression = 'gzip')
scrs_normal.write_h5ad(os.path.join(out_dir, "scrs_normal.h5ad"), compression = 'gzip')
scrs_tumor.write_h5ad(os.path.join(out_dir, "scrs_tumor.h5ad"), compression = 'gzip')